In [6]:
import os
print(os.getcwd())

C:\Users\ش\Desktop\img project


In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.image import load_img
import numpy as np

In [6]:

model_path = r"C:\Users\ش\Desktop\img project\emotion_model.keras" 
model = tf.keras.models.load_model(model_path)
TEST_DIR = r"C:/Users/ش/Desktop/img project/emotions_balanced/test"  

In [7]:
def load_test_images_and_labels(directory):
    image_paths = []
    labels = []
    for label in os.listdir(directory):
        label_dir = os.path.join(directory, label)
        if not os.path.isdir(label_dir):
            continue
        for fname in os.listdir(label_dir):
            image_paths.append(os.path.join(label_dir, fname))
            labels.append(label)
    return image_paths, labels

In [8]:
test_paths, test_labels_str = load_test_images_and_labels(TEST_DIR)

# Convert labels to integers using same encoder as training
le = LabelEncoder()
le.fit(test_labels_str)          # ensures same mapping as before
y_test_int = le.transform(test_labels_str)
y_test_cat = to_categorical(y_test_int, num_classes=6)

In [9]:
# Load and normalize images
x_test = []
for img_path in test_paths:
    img = load_img(img_path, color_mode='grayscale', target_size=(48,48))
    img_array = np.array(img) / 255.0
    x_test.append(img_array)
x_test = np.array(x_test).reshape(-1,48,48,1)

In [10]:
# ---------------------------
# 3. Predict on test set
# ---------------------------
y_pred_prob = model.predict(x_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = y_test_int


156/156 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step


In [11]:
# ---------------------------
# 4. Confusion Matrix & Per-class TP/TN/FP/FN
# ---------------------------
cm = confusion_matrix(y_true, y_pred)

# Build a DataFrame with detailed metrics
class_names = le.classes_  # ['anger','fear','happy','neutral','sad','surprise']
results = []

for i, name in enumerate(class_names):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    tn = cm.sum() - (tp + fp + fn)
    precision = tp / (tp + fp) if (tp+fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp+fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision+recall) > 0 else 0
    results.append([name, tp, tn, fp, fn, precision, recall, f1])

df_results = pd.DataFrame(results, columns=[
    'Class', 'TP', 'TN', 'FP', 'FN', 'Precision', 'Recall', 'F1-Score'
])

In [12]:


print("\n========== PER-CLASS METRICS ==========")
print(df_results.round(4))

# Overall accuracy
accuracy = (cm.trace()) / cm.sum()
print(f"\nOverall Accuracy: {accuracy:.4f}")

# full classification report
print("\n========== CLASSIFICATION REPORT ==========")
print(classification_report(y_true, y_pred, target_names=class_names))


========== PER-CLASS METRICS ==========
      Class   TP    TN   FP   FN  Precision  Recall  F1-Score
0     angry  485  3794  361  346     0.5733  0.5836    0.5784
1      fear  352  3867  288  479     0.5500  0.4236    0.4786
2     happy  667  3940  215  164     0.7562  0.8026    0.7788
3   neutral  513  3703  452  318     0.5316  0.6173    0.5713
4       sad  360  3776  379  471     0.4871  0.4332    0.4586
5  surprise  701  3942  213  130     0.7670  0.8436    0.8034

Overall Accuracy: 0.6173

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

       angry       0.57      0.58      0.58       831
        fear       0.55      0.42      0.48       831
       happy       0.76      0.80      0.78       831
     neutral       0.53      0.62      0.57       831
         sad       0.49      0.43      0.46       831
    surprise       0.77      0.84      0.80       831

    accuracy                           0.62      4986
   macro avg       0